In [ ]:
# OpenAI Careers Job Listings - Ashby

## 1. Project Overview

This project uses Python to automatically collect publicly available job listing information from a company's careers website. The scraped data will be transformed into a structured dataset, cleaned, analyzed, and visualized to identify patterns in hiring demand, job locations, and advertised compensation.


In [ ]:
# first import
import requests
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Retrieve the data

url = "https://api.ashbyhq.com/posting-api/job-board/openai?includeCompensation=true"
response = requests.get(url)
response.status_code

In [ ]:
# response

data = response.json()
type(data)

In [ ]:
data.keys()

In [ ]:
# jobs retrieved

len(data["jobs"])


In [ ]:
data["jobs"][0]

In [ ]:
# dataset structure

jobs = data["jobs"]

records = []

for job in jobs:
    compensation = job.get("compensation") or {}
    salary_components = compensation.get("summaryComponents") or []

    salary = next(
        (
            component
            for component in salary_components
            if component.get("compensationType") == "Salary"
        ),
        {}
    )

    salary_min = salary.get("minValue")
    salary_max = salary.get("maxValue")

    records.append({
        "Job Title": job.get("title"),
        "Department": job.get("department"),
        "Team": job.get("team"),
        "Employment Type": job.get("employmentType"),
        "Location": job.get("location"),
        "Workplace Type": job.get("workplaceType"),
        "Published Date": job.get("publishedAt"),
        "Salary Min": salary_min,
        "Salary Max": salary_max,
        "Job URL": job.get("jobUrl")
    })

df = pd.DataFrame(records)

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
# check for duplicates

df.duplicated().sum()
df["Job URL"].duplicated().sum()

In [ ]:
# convert published date

df["Published Date"] = pd.to_datetime(df["Published Date"])
df["Published Date"].dtype

In [ ]:
# salary midpoint

df["Salary Midpoint"] = (
    df["Salary Min"] + df["Salary Max"]
) / 2
df[["Salary Min", "Salary Max", "Salary Midpoint"]].head(10)

In [ ]:
# workplace type

df["Workplace Type"] = df["Workplace Type"].fillna("Not Specified")
df["Workplace Type"].value_counts()

In [ ]:
df["Salary Midpoint"].isna().sum()

In [ ]:
# clean data check

df.info()
df.head()

In [ ]:
df.describe(include="all")

In [ ]:
df.to_csv("openai_job_listings.csv", index=False)

In [ ]:
# 2. Exploratory Analysis

In [ ]:
## jobs by department

department_counts = (
    df["Department"]
    .value_counts()
    .reset_index()
)

department_counts.columns = ["Department", "Job Count"]

department_counts

In [ ]:
department_counts.head(10)

In [ ]:
## jobs by location

location_counts = (
    df["Location"]
    .value_counts()
    .reset_index()
)

location_counts.columns = ["Location", "Job Count"]

location_counts.head(15)

In [ ]:
## workplace type

workplace_counts = (
    df["Workplace Type"]
    .value_counts()
    .reset_index()
)

workplace_counts.columns = ["Workplace Type", "Job Count"]

workplace_counts

In [ ]:
## Salary analysis

salary_df = df.dropna(
    subset=["Salary Min", "Salary Max"]
).copy()

salary_df.shape

In [ ]:
salary_df[
    ["Salary Min", "Salary Max", "Salary Midpoint"]
].describe()

In [ ]:
## average salary by department

salary_by_department = (
    salary_df
    .groupby("Department")["Salary Midpoint"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)

salary_by_department

In [ ]:
## top 10 highest paying job listing

top_salary_jobs = (
    salary_df[
        ["Job Title", "Department", "Location", "Salary Min",
         "Salary Max", "Salary Midpoint"]
    ]
    .sort_values("Salary Midpoint", ascending=False)
    .head(10)
)

top_salary_jobs

In [ ]:
## job posting activity

df["Published Year"] = df["Published Date"].dt.year
year_counts = (
    df["Published Year"]
    .value_counts()
    .sort_index()
)

year_counts

In [ ]:
monthly_counts = (
    df.set_index("Published Date")
    .resample("ME")
    .size()
)

monthly_counts.tail(15)

In [ ]:
print(df.columns.tolist())

In [ ]:
df.shape

In [ ]:
df.info()